# Finance Agent — Run Comparison
Analyzes and compares backtest runs stored in `data/runs.db`.

**Sections**
1. Setup & connection
2. Summary table of all runs
3. Portfolio value evolution (time series)
4. ROI comparison bar chart
5. Trade analysis (BUY/SELL frequency per ticker)
6. Config vs ROI scatter

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is importable
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DB_PATH = Path("../data/runs.db")
assert DB_PATH.exists(), f"Database not found at {DB_PATH.resolve()}. Run a simulation first."
print(f"Connected to: {DB_PATH.resolve()}")

In [ ]:
# ── Helper ──────────────────────────────────────────────────────────────────
def query(sql: str, params=()) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

## 1. Summary of all runs

In [ ]:
runs = query("""
    SELECT
        run_id,
        substr(created_at, 1, 19)                       AS created_at,
        tickers,
        initial_capital,
        years,
        interval,
        final_value,
        ROUND(roi_pct, 2)                               AS roi_pct,
        ROUND(bh_roi_pct, 2)                            AS bh_roi_pct,
        ROUND(roi_pct - bh_roi_pct, 2)                  AS alpha_pp,
        dividends_received,
        num_trades,
        cash_remaining
    FROM runs
    ORDER BY created_at DESC
""")

print(f"{len(runs)} run(s) found")
runs.style \
    .format({
        "initial_capital":    "${:,.0f}",
        "final_value":        "${:,.2f}",
        "roi_pct":            "{:+.2f}%",
        "bh_roi_pct":         "{:+.2f}%",
        "alpha_pp":           "{:+.2f} pp",
        "dividends_received": "${:,.2f}",
        "cash_remaining":     "${:,.2f}",
    }) \
    .background_gradient(subset=["roi_pct", "bh_roi_pct"], cmap="RdYlGn") \
    .background_gradient(subset=["alpha_pp"], cmap="coolwarm")


## 2. Portfolio value evolution (time series)

In [ ]:
snapshots = query("""
    SELECT s.run_id, s.date, s.total_value,
           r.initial_capital, r.tickers, r.interval
    FROM portfolio_snapshots s
    JOIN runs r USING (run_id)
    ORDER BY s.run_id, s.date
""")
snapshots["date"] = pd.to_datetime(snapshots["date"])

fig, ax = plt.subplots()
for run_id, grp in snapshots.groupby("run_id"):
    label = f"{run_id}\n({grp['tickers'].iloc[0]}, {grp['interval'].iloc[0]})"
    ic = grp["initial_capital"].iloc[0]
    ax.plot(grp["date"], grp["total_value"], marker="", label=label, linewidth=1.5)
    ax.axhline(ic, linestyle="--", alpha=0.3, linewidth=0.8)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_title("Portfolio Value Over Time by Run")
ax.set_xlabel("Date")
ax.set_ylabel("Total Value (USD)")
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
plt.show()

## 3. ROI % per run

In [ ]:
has_bh = "bh_roi_pct" in runs.columns and runs["bh_roi_pct"].notna().any()
n_runs = len(runs)
width = 0.35
x = range(n_runs)

fig, ax = plt.subplots(figsize=(max(7, n_runs * 2), 5))

# Agent bars
agent_colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in runs["roi_pct"]]
bars_agent = ax.bar(
    [i - (width / 2 if has_bh else 0) for i in x],
    runs["roi_pct"],
    width if has_bh else 0.6,
    label="Agent",
    color=agent_colors,
    edgecolor="white",
    linewidth=0.5,
)

if has_bh:
    bh_colors = ["#27ae60" if v >= 0 else "#c0392b" for v in runs["bh_roi_pct"].fillna(0)]
    bars_bh = ax.bar(
        [i + width / 2 for i in x],
        runs["bh_roi_pct"].fillna(0),
        width,
        label="Buy & Hold",
        color=bh_colors,
        alpha=0.6,
        edgecolor="white",
        linewidth=0.5,
        hatch="//",
    )
    for bar, val in zip(bars_bh, runs["bh_roi_pct"].fillna(0)):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + (0.2 if val >= 0 else -0.8),
            f"{val:+.1f}%",
            ha="center", va="bottom" if val >= 0 else "top",
            fontsize=8, color="gray",
        )

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")

for bar, val in zip(bars_agent, runs["roi_pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + (0.3 if val >= 0 else -1.0),
        f"{val:+.1f}%",
        ha="center", va="bottom" if val >= 0 else "top",
        fontsize=9, fontweight="bold",
    )

ax.set_title("Agent ROI vs Buy & Hold benchmark")
ax.set_ylabel("ROI (%)")
ax.set_xlabel("Run ID")
ax.set_xticks(list(x))
ax.set_xticklabels(runs["run_id"], rotation=30, ha="right")
if has_bh:
    ax.legend()
fig.tight_layout()
plt.show()


## 4. Trade analysis — BUY/SELL frequency per ticker

In [ ]:
trades = query("""
    SELECT t.run_id, t.ticker, t.action, COUNT(*) AS count
    FROM trades t
    GROUP BY t.run_id, t.ticker, t.action
    ORDER BY t.run_id, t.ticker, t.action
""")

if trades.empty:
    print("No trades found.")
else:
    pivot = trades.pivot_table(
        index=["run_id", "ticker"], columns="action", values="count", fill_value=0
    ).reset_index()

    fig, ax = plt.subplots(figsize=(14, 5))
    x_labels = pivot["run_id"] + " / " + pivot["ticker"]
    x = range(len(x_labels))
    width = 0.35

    if "BUY" in pivot.columns:
        ax.bar([i - width/2 for i in x], pivot["BUY"], width, label="BUY", color="#2ecc71")
    if "SELL" in pivot.columns:
        ax.bar([i + width/2 for i in x], pivot["SELL"], width, label="SELL", color="#e74c3c")

    ax.set_xticks(list(x))
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_title("Trade Count per Ticker per Run")
    ax.set_ylabel("# Trades")
    ax.legend()
    fig.tight_layout()
    plt.show()
    display(pivot)

## 5. Config vs Result — Initial capital vs ROI %

In [ ]:
if len(runs) < 2:
    print("Necesitás al menos 2 runs para ver el scatter. Ejecutá más simulaciones.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Capital vs ROI
    ax = axes[0]
    scatter = ax.scatter(
        runs["initial_capital"], runs["roi_pct"],
        c=runs["years"], cmap="viridis", s=80, edgecolors="white", linewidth=0.5
    )
    for _, row in runs.iterrows():
        ax.annotate(row["run_id"][:12], (row["initial_capital"], row["roi_pct"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Initial Capital (USD)")
    ax.set_ylabel("ROI (%)")
    ax.set_title("Initial Capital vs ROI")
    fig.colorbar(scatter, ax=ax, label="Years")

    # Years vs ROI
    ax2 = axes[1]
    ax2.scatter(
        runs["years"], runs["roi_pct"],
        c=runs["num_trades"], cmap="plasma", s=80, edgecolors="white", linewidth=0.5
    )
    for _, row in runs.iterrows():
        ax2.annotate(row["run_id"][:12], (row["years"], row["roi_pct"]),
                     fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax2.set_xlabel("Simulation Years")
    ax2.set_ylabel("ROI (%)")
    ax2.set_title("Simulation Years vs ROI")

    fig.tight_layout()
    plt.show()

## 6. Deep dive — select specific runs to compare

In [ ]:
# Modify this list with the run IDs you want to compare in detail
COMPARE_RUNS = runs["run_id"].tolist()  # default: all runs

if COMPARE_RUNS:
    placeholders = ",".join(["?" * len(COMPARE_RUNS)])
    detail_snaps = query(
        f"SELECT run_id, date, total_value, cash FROM portfolio_snapshots "
        f"WHERE run_id IN ({','.join(['?']*len(COMPARE_RUNS))}) ORDER BY run_id, date",
        tuple(COMPARE_RUNS)
    )
    detail_snaps["date"] = pd.to_datetime(detail_snaps["date"])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=False)

    for run_id, grp in detail_snaps.groupby("run_id"):
        ax1.plot(grp["date"], grp["total_value"], label=run_id, linewidth=1.5)
        ax2.plot(grp["date"], grp["cash"], label=run_id, linewidth=1.5, linestyle="--")

    ax1.set_title("Total Portfolio Value")
    ax1.set_ylabel("USD")
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax1.legend(fontsize=8)

    ax2.set_title("Cash Position")
    ax2.set_ylabel("USD")
    ax2.set_xlabel("Date")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax2.legend(fontsize=8)

    fig.tight_layout()
    plt.show()